══════════════════════════════════════════════════════════════
Beyond Distance: Predicting U.S. Airfare Through Market Structure
Datathon 2025
══════════════════════════════════════════════════════════════

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
FILE_PATH = 'airline_ticket_dataset.xlsx'

──────────────────────────────────────────────────────────────
PART 1
(Team Member: Carina Zhang)
──────────────────────────────────────────────────────────────

In [12]:
%load_ext rpy2.ipython

In [22]:
%%R
library(readxl)
library(tidyverse)
library(ggplot2)
library(dplyr)
library(httr)

url <- "https://raw.githubusercontent.com/auDreyLuo274/Predicting-U.S.-Airfare-Through-Market-Structure/main/airline_ticket_dataset.xlsx"
GET(url, write_disk("airline_ticket_dataset.xlsx", overwrite = TRUE))

airline_ticket_dataset <- read_excel("airline_ticket_dataset.xlsx")
airline_tickets <- airline_ticket_dataset

airline_tickets <- airline_tickets |> mutate(flight_length = case_when(nsmiles < 930 ~ 'short', nsmiles < 1860 & nsmiles > 930~ 'medium', nsmiles >1860 ~ 'long'))
scatterplot_demand <- airline_tickets |> ggplot(aes(x = log(passengers), y = log(fare))) +
  geom_point()
scatterplot_demand

scatterplot_distance <- airline_tickets |> ggplot(aes(x = nsmiles, y = fare)) +
  geom_point()
scatterplot_distance

scatterplot_market_share <- airline_tickets |> ggplot(aes(x = 1- large_ms, y = fare)) +
  geom_point() + facet_wrap(~flight_length)
scatterplot_market_share

airline_tickets <- airline_tickets |> mutate(price_diff = fare_lg - fare_low)
scatterplot_price_diff <- airline_tickets |> ggplot(aes(x = price_diff, y = fare)) +
  geom_point()
scatterplot_price_diff

scatterplot_lf_ms <- airline_tickets |> ggplot(aes(x = lf_ms, y = fare)) +
  geom_point()

airline_tickets <- airline_tickets |> mutate(totalPrem = (TotalPerPrem_city2 * TotalFaredPax_city2 + TotalPerPrem_city1 * TotalFaredPax_city1) / TotalFaredPax_city1 + TotalFaredPax_city2)
scatterplot_Prem <- airline_tickets |> ggplot(aes(x = totalPrem, y = fare)) +
  geom_point()
scatterplot_Prem

lm_distance <- lm(fare~nsmiles, data = airline_tickets)
summary(lm_distance)

lm_competition <- lm(fare~large_ms + price_diff + lf_ms, data = airline_tickets)
summary(lm_competition)

lm_endpoint <- lm(fare~TotalPerLFMkts_city1 + TotalPerLFMkts_city2, data = airline_tickets)
summary(lm_endpoint)

lm_endpoint <- lm(fare~TotalPerLFMkts_city1 + TotalPerLFMkts_city2 + nsmiles + passengers + large_ms + lf_ms + TotalFaredPax_city1 + TotalFaredPax_city2 + TotalPerPrem_city1 + TotalPerPrem_city2 + carrier_lg + carrier_low + city1 + city2, data = airline_tickets)
summary(lm_endpoint)

summary_table <- airline_tickets |>
  summarize(
    mean_large_ms = mean(large_ms, na.rm = TRUE),

    mean_large_hub_ms = mean(
      large_ms[
        city1 %in% c("New York City, NY (Metropolitan Area)",
                     "Los Angeles, CA (Metropolitan Area)",
                     "San Francisco, CA (Metropolitan Area)",
                     "Washington, DC (Metropolitan Area)",
                     "Chicago, IL", "Dallas/Fort Worth, TX",
                     "Atlanta, GA (Metropolitan Area)",
                     "Miami, FL (Metropolitan Area)",
                     "Denver, CO")&
        city2 %in% c("New York City, NY (Metropolitan Area)",
                     "Los Angeles, CA (Metropolitan Area)",
                     "San Francisco, CA (Metropolitan Area)",
                     "Washington, DC (Metropolitan Area)",
                     "Chicago, IL", "Dallas/Fort Worth, TX",
                     "Atlanta, GA (Metropolitan Area)",
                     "Miami, FL (Metropolitan Area)",
                     "Denver, CO")
      ],
      na.rm = TRUE
    )
  )
print(summary_table)

summary_table2 <- airline_tickets |>
  summarize(
    median_distance = median(nsmiles, na.rm = TRUE),

    fare_per_mile_short = mean(
      fare[nsmiles < median(nsmiles, na.rm = TRUE)] /
      nsmiles[nsmiles < median(nsmiles, na.rm = TRUE)],
      na.rm = TRUE
    ),

    fare_per_mile_long = mean(
      fare[nsmiles >= median(nsmiles, na.rm = TRUE)] /
      nsmiles[nsmiles >= median(nsmiles, na.rm = TRUE)],
      na.rm = TRUE
    )
  )
print(summary_table2)

# A tibble: 1 × 2
  mean_large_ms mean_large_hub_ms
          <dbl>             <dbl>
1         0.566             0.433
# A tibble: 1 × 3
  median_distance fare_per_mile_short fare_per_mile_long
            <dbl>               <dbl>              <dbl>
1             936               0.392              0.185


──────────────────────────────────────────────────────────────
PART 2
(Team Member: )
──────────────────────────────────────────────────────────────

[]

──────────────────────────────────────────────────────────────
PART 3: SHAP Analysis & Feature Importance
(Team Member: Audrey Luo)
──────────────────────────────────────────────────────────────

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
# ─────────────────────────────────────────────
# 1. LOAD DATASET
# ─────────────────────────────────────────────
file_path = '/Users/luozihan/Desktop/Airline Tickets/airline_ticket_dataset.xlsx'
df = pd.read_excel(file_path)

In [ ]:
# ─────────────────────────────────────────────
# 2. FEATURE ENGINEERING
#    Layer 1: Distance + Demand (baseline)
#    Layer 2: + Competition
#    Layer 3: + Hub characteristics (Full Model)
# ─────────────────────────────────────────────
features = [
    # Layer 1 — Baseline
    'nsmiles',
    'passengers',
    # Layer 2 — Competition
    'large_ms',
    'lf_ms',
    'fare_low',
    # Layer 3 — Hub characteristics (both endpoints)
    'TotalFaredPax_city1',
    'TotalPerLFMkts_city1',
    'TotalPerPrem_city1',
    'TotalFaredPax_city2',
    'TotalPerLFMkts_city2',
    'TotalPerPrem_city2',
]

In [ ]:
X = df[features].fillna(0)
y = df['fare']

In [ ]:
# ─────────────────────────────────────────────
# 3. TRAIN / TEST SPLIT
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# ─────────────────────────────────────────────
# 4. TRAIN XGBOOST MODEL
# ─────────────────────────────────────────────
print("Training XGBoost model...")
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

In [ ]:
# ─────────────────────────────────────────────
# 5. MODEL EVALUATION
# ─────────────────────────────────────────────
y_pred = model.predict(X_test)
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

In [ ]:
print("\n── Model Performance on Test Data ──")
print(f"  R²   : {r2:.4f}")
print(f"  MAE  : ${mae:.2f}")
print(f"  RMSE : ${rmse:.2f}")

In [ ]:
# ─────────────────────────────────────────────
# 6. SHAP VALUES  (unified old API — most stable)
# ─────────────────────────────────────────────
sample_size   = int(len(X_test) * 0.8)
X_test_sample = X_test.sample(n=sample_size, random_state=42)

In [ ]:
print(f"\nCalculating SHAP values for {sample_size} random test samples...")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_sample)

In [ ]:
# ─────────────────────────────────────────────
# 7. PLOT 1 — Summary Plot (importance + direction)
# ─────────────────────────────────────────────
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, show=False)
plt.title("SHAP: Feature Importance & Direction of Impact on Airfare", fontsize=14)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → shap_summary.png")

In [ ]:
# ─────────────────────────────────────────────
# 8. PLOT 2 — Bar Plot (clean ranking for presentation)
# ─────────────────────────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_sample, plot_type="bar", show=False)
plt.title("SHAP: Mean Absolute Feature Importance Ranking", fontsize=14)
plt.tight_layout()
plt.savefig("shap_bar.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → shap_bar.png")

In [ ]:
# ─────────────────────────────────────────────
# 9. PLOT 3 — Dependence Plot (interaction effect)
# ─────────────────────────────────────────────
plt.figure(figsize=(10, 6))
shap.dependence_plot(
    "large_ms",
    shap_values,
    X_test_sample,
    interaction_index="nsmiles",
    show=False,
)
plt.title("SHAP Dependence: Market Concentration × Distance\n"
          "(colour = route distance — reveals monopoly premium on short routes)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# 10. EXPORT MODEL DATA FOR THE WEB TOOL
# ─────────────────────────────────────────────
feature_stats = {
    feat: {
        "min":    float(X[feat].min()),
        "max":    float(X[feat].max()),
        "mean":   float(X[feat].mean()),
        "median": float(X[feat].median()),
    }
    for feat in features
}

In [ ]:
mean_shap = dict(zip(features, np.abs(shap_values).mean(axis=0)))
feature_stats_out = {
    "feature_stats": feature_stats,
    "mean_shap": {k: float(v) for k, v in mean_shap.items()},
    "model_r2":   round(r2, 4),
    "model_mae":  round(mae, 2),
    "model_rmse": round(rmse, 2),
}

In [ ]:
with open("model_stats.json", "w") as f:
    json.dump(feature_stats_out, f, indent=2)

In [ ]:
import pickle
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

In [ ]:
# ─────────────────────────────────────────────
# ── Ablation Study: Model Without fare_low ──
# ─────────────────────────────────────────────
features_no_farelow = [f for f in features if f != 'fare_low']

In [ ]:
X_abl       = df[features_no_farelow].fillna(0)
X_abl_train, X_abl_test, y_abl_train, y_abl_test = train_test_split(
    X_abl, y, test_size=0.2, random_state=42
)

In [ ]:
print("\nTraining ablation model (no fare_low)...")
model_abl = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model_abl.fit(X_abl_train, y_abl_train)

In [ ]:
y_abl_pred = model_abl.predict(X_abl_test)
r2_abl   = r2_score(y_abl_test, y_abl_pred)
mae_abl  = mean_absolute_error(y_abl_test, y_abl_pred)
rmse_abl = np.sqrt(mean_squared_error(y_abl_test, y_abl_pred))

In [ ]:
print("\n── Ablation Study: Model Comparison ──")
print(f"{'Metric':<8}  {'Full Model':>12}  {'No fare_low':>12}  {'Δ':>10}")
print(f"{'R²':<8}  {r2:>12.4f}  {r2_abl:>12.4f}  {r2_abl - r2:>+10.4f}")
print(f"{'MAE':<8}  ${mae:>11.2f}  ${mae_abl:>11.2f}  ${mae_abl - mae:>+9.2f}")
print(f"{'RMSE':<8}  ${rmse:>11.2f}  ${rmse_abl:>11.2f}  ${rmse_abl - rmse:>+9.2f}")

In [ ]:
sample_size_abl   = int(len(X_abl_test) * 0.8)
X_abl_test_sample = X_abl_test.sample(n=sample_size_abl, random_state=42)

In [ ]:
print(f"\nCalculating SHAP values for ablation model ({sample_size_abl} samples)...")
explainer_abl   = shap.TreeExplainer(model_abl)
shap_values_abl = explainer_abl.shap_values(X_abl_test_sample)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_abl, X_abl_test_sample, plot_type="bar", show=False)
plt.title(
    "SHAP: Feature Importance — Model Without fare_low\n"
    f"R²={r2_abl:.4f}  MAE=${mae_abl:.2f}  RMSE=${rmse_abl:.2f}  "
    f"(vs. full model R²={r2:.4f})",
    fontsize=12,
)
plt.tight_layout()
plt.savefig("shap_bar_no_farelow.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → shap_bar_no_farelow.png")